# Adaptive Attack Evaluation


In [1]:

import os, sys, torch
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
sys.path.insert(0, os.path.abspath('.'))

from src.config import DEVICE, RESULTS_DIR, CHECKPOINT_DIR
from src.model import get_model
from src.data_loader import build_partition_indices, get_partition
from src.attack import load_patch, apply_patch
from src.gradcam_defense import GradCAM
from src.feature_squeeze import load_threshold

print(f"Device: {DEVICE}")
model        = get_model()
patch        = load_patch()
partitions   = build_partition_indices()
gradcam      = GradCAM(model, target_layer_name='layer4')
fs_threshold = load_threshold()
print(f"FS threshold: {fs_threshold:.3f}")


[config] Using device: mps
Device: mps
Loading pretrained ResNet-50 …
Ready
Patch loaded  shape: (3, 89, 89)
Threshold loaded: 0.150
FS threshold: 0.150


## Step 1: Train adaptive patch

In [11]:
import os, torch
_path = 'results/patches/patch_adaptive_89px.pt'
_stale_path = 'results/patches/patch_adaptive.pt'  # old, wrongly-shaped file

def _is_valid_89px(p):
    return os.path.exists(p) and tuple(torch.load(p, map_location=DEVICE, weights_only=False).shape) == (3, 89, 89)

if os.path.exists(_stale_path) and not os.path.exists(_path):
    # One-time migration check: the old filename may hold a 44px patch left
    # over from the nb3/patch_adaptive.pt collision. Never trust it blindly.
    _old = torch.load(_stale_path, map_location=DEVICE, weights_only=False)
    if tuple(_old.shape) == (3, 89, 89):
        print(f"[MIGRATE] {_stale_path} is actually 89px -- copying to new path.")
        torch.save(_old, _path)
    else:
        print(f"[STALE] {_stale_path} has shape {tuple(_old.shape)} (expected 89px) -- ignoring, will retrain.")

if _is_valid_89px(_path):
    print("[SKIP] patch_adaptive_89px.pt exists and is 89px -- loading.")
    patch_adaptive = torch.load(_path, map_location=DEVICE, weights_only=False)
    print(f"Adaptive patch shape: {tuple(patch_adaptive.shape)}")
else:
    print("[RUN] Training 89px adaptive patch (~90 min on M3 Max)...")
    from src.adaptive_attack import train_adaptive_patch
    from src.data_loader import get_loaders
    from src.model import get_base_model
    train_loader, _, _ = get_loaders()
    patch_adaptive = train_adaptive_patch(
        model=get_base_model(), data_loader=train_loader,
        steps=1000, lambda_kp=0.5, lambda_fs=0.3, save=True,
    )
    assert tuple(patch_adaptive.shape) == (3, 89, 89), (
        f"Trained adaptive patch has shape {tuple(patch_adaptive.shape)}, expected 89px -- "
        f"check src.config.PATCH_SIZE_PX was not overridden by a prior sweep run."
    )
    print("Adaptive patch trained and saved.")

[SKIP] patch_adaptive_89px.pt exists and is 89px -- loading.
Adaptive patch shape: (3, 89, 89)


## Step 2: Load eval data

In [12]:

# Load final_eval partition (same 1000 images as Table II)
images_clean, labels = get_partition("final_eval", partitions)
images_patched, _    = apply_patch(images_clean.clone(), patch, augment=False)
print(f"Loaded {images_clean.size(0)} eval images")


Loaded 1000 eval images


## Step 3: Evaluate standard vs adaptive patch

In [2]:

import os, torch
from src.pipeline import HybridDefense
from src.stats import bootstrap_ci

CHECKPOINT = 'results/adaptive_eval_checkpoint.pt'
if os.path.exists(CHECKPOINT):
    _ck_probe = torch.load(CHECKPOINT, weights_only=False)
    if not _ck_probe.get('_patch_shape_89px_verified', False):
        print(f"[STALE] {CHECKPOINT} predates the 89px fix -- deleting and re-evaluating.")
        os.remove(CHECKPOINT)

def _is_complete(n=300):
    if not os.path.exists(CHECKPOINT): return False
    ck = torch.load(CHECKPOINT, weights_only=False)
    return (ck.get('standard',{}).get('done',0) >= n and
            ck.get('adaptive',{}).get('done',0) >= n)

if _is_complete():
    ck        = torch.load(CHECKPOINT, weights_only=False)
    std_res   = ck['standard']
    adp_res   = ck['adaptive']
    delta_pp  = std_res['recovered_acc'] - adp_res['recovered_acc']
    print("[SKIP] Adaptive evaluation already complete.")
    print(f"  Standard recovered: {std_res['recovered_acc']:.1f}%")
    print(f"  Adaptive recovered: {adp_res['recovered_acc']:.1f}%")
    print(f"  Delta: {delta_pp:+.1f}pp")
else:
    _avail    = min(images_clean.size(0), labels.size(0))
    N_EVAL    = min(300, _avail)
    eval_imgs = images_clean[:N_EVAL].cpu()
    eval_lbls = labels[:N_EVAL].cpu()
    assert eval_lbls.size(0) == N_EVAL
    print(f"Evaluating on {N_EVAL} images...")

    assert tuple(patch_adaptive.shape) == (3, 89, 89), (
        f"patch_adaptive has shape {tuple(patch_adaptive.shape)}, expected 89px -- "
        f"do not evaluate a size mismatch against the standard 89px patch."
    )
    if not os.path.exists(CHECKPOINT):
        torch.save({'_patch_shape_89px_verified': True}, CHECKPOINT)

    defense = HybridDefense(model, gradcam, fs_threshold, kp_method='orb')

    def eval_patch_safe(p, name, imgs, lbls, key):
        n   = imgs.size(0)
        ck  = torch.load(CHECKPOINT, weights_only=False) if os.path.exists(CHECKPOINT) else {}
        if ck.get(key, {}).get('done', 0) >= n:
            print(f"  {name}: complete in checkpoint.")
            return ck[key]
        start     = ck.get(key, {}).get('done', 0)
        recovered = ck.get(key, {}).get('recovered', 0)
        from tqdm import tqdm
        for i in tqdm(range(start, n), desc=f"  {name}"):
            img = imgs[i].unsqueeze(0).to(DEVICE)
            patched_img, _ = apply_patch(img, p.to(DEVICE), augment=False)
            res = defense.defend(patched_img.squeeze(0), true_label=lbls[i].item())
            if res["correct"]: recovered += 1
            if DEVICE.type == 'mps': torch.mps.empty_cache()
            if (i+1) % 50 == 0:
                ck[key] = {"recovered": recovered, "done": i+1,
                            "recovered_acc": round(100*recovered/(i+1), 2)}
                torch.save(ck, CHECKPOINT)
        result = {"recovered": recovered, "done": n,
                   "recovered_acc": round(100*recovered/n, 2)}
        ck[key] = result; torch.save(ck, CHECKPOINT)
        return result

    std_res = eval_patch_safe(patch,          "Standard", eval_imgs, eval_lbls, "standard")
    adp_res = eval_patch_safe(patch_adaptive, "Adaptive", eval_imgs, eval_lbls, "adaptive")

    with torch.no_grad():
        clean_p   = model(eval_imgs.to(DEVICE)).argmax(1).cpu()
        if DEVICE.type == 'mps': torch.mps.empty_cache()
        std_p_img, _ = apply_patch(eval_imgs.to(DEVICE), patch.to(DEVICE), augment=False)
        std_preds = model(std_p_img).argmax(1).cpu(); del std_p_img
        if DEVICE.type == 'mps': torch.mps.empty_cache()
        adp_p_img, _ = apply_patch(eval_imgs.to(DEVICE), patch_adaptive.to(DEVICE), augment=False)
        adp_preds = model(adp_p_img).argmax(1).cpu(); del adp_p_img
        if DEVICE.type == 'mps': torch.mps.empty_cache()

    ok = (clean_p == eval_lbls)
    std_asr = ((ok) & (std_preds != eval_lbls)).float().sum() / ok.sum()
    adp_asr = ((ok) & (adp_preds != eval_lbls)).float().sum() / ok.sum()
    delta_pp = std_res["recovered_acc"] - adp_res["recovered_acc"]

    std_fail = 1.0 - std_res["recovered"] / N_EVAL
    adp_fail = 1.0 - adp_res["recovered"] / N_EVAL

    print("\n" + "="*70)
    print(f"  n={N_EVAL}  SE~{100*(0.5/N_EVAL**0.5):.1f}pp")
    print(f"{'':35s} {'Oblivious':>15} {'Adaptive':>15}")
    print("-"*70)
    print(f"  Undefended ASR:         {std_asr*100:>14.1f}% {adp_asr*100:>14.1f}%")
    print(f"  Post-defense fail rate: {std_fail*100:>14.1f}% {adp_fail*100:>14.1f}%")
    print(f"  Defense degradation:    {'—':>15} {(adp_fail-std_fail)*100:>+14.1f}pp")
    print(f"  Adaptive cost (ASR):    {'—':>15} {(adp_asr-std_asr)*100:>+14.1f}pp")
    print("="*70)

    se = 100 * (0.5 / N_EVAL ** 0.5)
    asr_delta = (adp_asr - std_asr) * 100        # positive = adaptive is MORE potent undefended
    fail_delta = (adp_fail - std_fail) * 100      # positive = defense does WORSE on adaptive

    def _sig(delta, se, n_se=2.0):
        return abs(delta) >= n_se * se

    print(f"\nSection IV-G framing (n={N_EVAL}, SE~{se:.1f}pp, "
          f"significance threshold ~{2*se:.1f}pp):")

    if asr_delta > 0:
        print(f"  Undefended ASR was {abs(asr_delta):.1f}pp HIGHER for the adaptive patch "
              f"({std_asr*100:.1f}% -> {adp_asr*100:.1f}%)"
              f"{' [not statistically significant]' if not _sig(asr_delta, se) else ''}.")
    else:
        print(f"  Adaptive patch sacrificed {abs(asr_delta):.1f}pp of undefended ASR "
              f"({std_asr*100:.1f}% -> {adp_asr*100:.1f}%) to attempt to evade the defense"
              f"{' [not statistically significant]' if not _sig(asr_delta, se) else ''}.")

    if fail_delta > 0:
        print(f"  Post-defense failure rate was {abs(fail_delta):.1f}pp WORSE under the "
              f"adaptive patch ({std_fail*100:.1f}% -> {adp_fail*100:.1f}%)"
              f"{' [not statistically significant]' if not _sig(fail_delta, se) else ''}.")
    else:
        print(f"  Post-defense failure rate was {abs(fail_delta):.1f}pp BETTER under the "
              f"adaptive patch ({std_fail*100:.1f}% -> {adp_fail*100:.1f}%) -- the hybrid "
              f"defense was not degraded by this adaptive attack"
              f"{' [not statistically significant]' if not _sig(fail_delta, se) else ''}.")

    # Proper bootstrap CI on each fail rate, using the actual recovered/done
    # counts already sitting in the checkpoint -- no extra experiment needed.
    std_fail_lo, std_fail_hi = bootstrap_ci(std_res['done'] - std_res['recovered'], std_res['done'])
    adp_fail_lo, adp_fail_hi = bootstrap_ci(adp_res['done'] - adp_res['recovered'], adp_res['done'])
    print(f"\n  Post-defense fail rate 95% CI: oblivious [{std_fail_lo:.1f}, {std_fail_hi:.1f}]%, "
          f"adaptive [{adp_fail_lo:.1f}, {adp_fail_hi:.1f}]%")
    ci_overlap = not (adp_fail_hi < std_fail_lo or std_fail_hi < adp_fail_lo)
    print(f"  CIs {'OVERLAP -- no significant difference' if ci_overlap else 'DO NOT overlap -- difference is likely real'}.")

    if not _sig(fail_delta, se):
        print(f"\n  Bottom line: at n={N_EVAL}, no statistically significant defense "
              f"degradation was detected under the adaptive attack. Consider re-running ")


[SKIP] Adaptive evaluation already complete.
  Standard recovered: 87.3%
  Adaptive recovered: 88.3%
  Delta: -1.0pp
